# EDA II: tiempo, espacio y preprocesamiento

Acompaña las sub-pestañas **Temporal, Espacial, Espacio-temporal y Preprocesamiento** del dashboard.

In [1]:
import os  # Rutas
import sys  # Configuración del intérprete
sys.path.insert(0, os.path.abspath(".."))  # Carpeta entregable2 (paquete tablero)
sys.path.insert(0, os.path.abspath("../.."))  # Raíz del proyecto (paquete src)
sys.path.insert(0, os.path.abspath("../../entregable1"))  # Paquete e1 (código fuente que se muestra)
import warnings  # Control de avisos
warnings.filterwarnings("ignore")  # Oculta avisos de versiones
import numpy as np  # Cálculo numérico
import pandas as pd  # Tablas
from IPython.display import HTML, Markdown, display  # Incrustar figuras y texto
from tablero import figuras as fg  # Las mismas figuras del dashboard
from tablero.datos import A, M, MOD, FEATURES, curvas, metricas_umbral, rezago_espacial  # Los mismos artefactos del dashboard
pd.set_option("display.width", 160)  # Ancho de las tablas
pd.set_option("display.max_columns", 30)  # Columnas visibles
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")  # Cuatro decimales


def mostrar(fig):  # Muestra una figura del dashboard como HTML interactivo
    """Pone el fondo oscuro de las tarjetas del dashboard y carga Plotly desde internet."""
    fig.update_layout(paper_bgcolor=fg.SUPERFICIE, plot_bgcolor=fg.SUPERFICIE, margin=dict(l=40, r=20, t=50, b=40))  # Fondo oscuro
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False, config={"displaylogo": False})))  # Figura interactiva
t = A["temporal"]  # Componente temporal
pd.Series({"fuerza de la tendencia": t["fuerza"]["tendencia"], "fuerza estacional": t["fuerza"]["estacional"],
           "municipios con ADF significativo": t["mun_adf"], "municipios con KPSS significativo": t["mun_kpss"]}).to_frame("valor")

,valor
fuerza de la tendencia,0.6610
fuerza estacional,0.0214
municipios con ADF significativo,0.6223
municipios con KPSS significativo,0.4867


## Componente temporal

In [2]:
mostrar(fg.fig_serie(t["nacional"], t["mensual"], A["contexto"]["anual"].loc[:2021], "semana"))  # Serie semanal
mostrar(fg.fig_estacionalidad(t["calendario"]))  # Estacionalidad
mostrar(fg.fig_stl(t["stl"]))  # STL

**Interpretación.** La serie la dominan ondas de varios años (fuerza de la tendencia 0,66) y la estacionalidad anual es casi nula (0,02): la información útil está en el nivel reciente de la serie.

:::{admonition} Detalle: fuerza de la tendencia y de la estacionalidad (Wang, Smith y Hyndman, 2006)
Con la descomposición STL $y_t = T_t + S_t + R_t$: $F_T = \max\big(0, 1 - \frac{\text{Var}(R_t)}{\text{Var}(T_t + R_t)}\big)$ y $F_S = \max\big(0, 1 - \frac{\text{Var}(R_t)}{\text{Var}(S_t + R_t)}\big)$.
:::

In [3]:
t["estacionariedad"]  # ADF (H0: raíz unitaria) y KPSS (H0: estacionaria)

,ADF estadístico,ADF p,KPSS estadístico,KPSS p
serie,,,,
Nivel: log casos,-2.5110,0.1128,0.2516,0.1000
Primera diferencia,-14.9251,0.0000,0.0506,0.1000


**Interpretación.** En nivel, ADF no rechaza la raíz unitaria y KPSS tampoco rechaza la estacionariedad: una serie muy persistente en la que ambas pruebas pierden potencia. La primera diferencia es estacionaria. Por municipio, el 62 % rechaza la raíz unitaria y el 49 % rechaza la estacionariedad.

In [4]:
mostrar(fg.fig_acf(t["acf"], t["pacf"], t["n"]))  # ACF y PACF
mostrar(fg.fig_ccf(t["ccf"]))  # Correlación cruzada
t["ccf"].abs().agg(["idxmax", "max"]).T.rename(columns={"idxmax": "rezago de máxima |r|", "max": "máxima |r|"})

,rezago de máxima |r|,máxima |r|
temp_anom,0.0000,0.0564
prec_anom,15.0000,0.0362
ndvi_anom,12.0000,0.0325
oni_anom,5.0000,0.1717


**Interpretación.** ACF persistente y PACF concentrada en los primeros rezagos: se usan rezagos de 1–3 semanas y medias de 4, 8 y 12. El clima local llega con rezago (lluvia y NDVI 12–15 semanas) y correlaciones pequeñas; el ONI es la señal más fuerte.

In [5]:
mostrar(fg.fig_regimenes(t["stl"], t["cortes"], t["nino"]))  # Regímenes
print("Puntos de cambio:", ", ".join(str(c.date()) for c in t["cortes"]))  # Fechas
display(t["deriva"])  # PSI
mostrar(fg.fig_deriva(t["concepto"]))  # Concept drift
t["concepto"]

Puntos de cambio: 2011-01-03, 2012-12-31, 2016-09-05, 2018-05-28, 2019-05-27, 2020-06-08


,PSI 2015–18 → 2019–21,PSI entrenamiento → 2022
lc_m4,0.0401,0.0068
brote_m8,0.0586,0.0033
ratio_umbral,0.0781,0.0328
temp_mean_m4,0.0029,0.0307
temp_anom_m8,0.0270,0.2010
prec_total_m4,0.0023,0.1049
ndvi_mean_m4,0.0016,0.0199
oni_anom,1.9548,7.4466
oni_l8,1.8518,7.5873
flujo_m4,0.0708,0.0639


,lc_m4,brote_m8,ratio_umbral,oni_l8,temp_anom_m8
2015,0.8307,0.8197,0.7170,0.5961,0.5531
2016,0.8640,0.8643,0.8110,0.6784,0.6437
2017,0.8304,0.7659,0.6764,0.5023,0.5017
2018,0.8862,0.8476,0.7895,0.6528,0.5509
2019,0.8831,0.8617,0.7897,0.4629,0.5085
2020,0.8708,0.8679,0.8263,0.6677,0.5574
2021,0.8915,0.8551,0.7821,0.5744,0.5171
2022,0.8937,0.8673,0.7861,0.5036,0.4587


**Interpretación.** Los regímenes siguen las epidemias y la caída de 2020 (pandemia). La deriva es baja para la historia de casos, moderada hacia 2022 en la temperatura anómala y la lluvia, y muy fuerte en el ONI; el concepto casos → brote es estable (AUC 0,83–0,89 cada año), el del clima no.

:::{admonition} Detalle: índice de estabilidad poblacional y segmentación binaria
$\text{PSI} = \sum_i (a_i - r_i)\ln(a_i/r_i)$ sobre los deciles de la referencia (< 0,1 estable; > 0,25 deriva fuerte). Los puntos de cambio se buscan dividiendo recursivamente la serie donde más baja la suma de cuadrados, con segmentos de al menos 52 semanas y una penalización tipo BIC $3 \cdot 2\hat\sigma^2 \ln n$, con $\hat\sigma^2$ la varianza del residuo de STL.
:::

In [6]:
mostrar(fg.fig_calor(t["calor"], t["calor_regiones"]))  # Panel municipio × semana

**Interpretación.** Ondas sincronizadas (franjas verticales) y focos locales: heterogeneidad entre municipios en nivel y en tiempo.

## Componente espacial

In [7]:
s = A["espacial"]  # Componente espacial
mapa = s["mapa"]  # Municipios
mostrar(fg.fig_cobertura(mapa, s["fuera"]))  # Cobertura
mostrar(fg.fig_mapa_valor(mapa, "log_incidencia", "log incidencia"))  # Incidencia
pd.Series(s["clark_evans"]).to_frame("Clark-Evans")

,Clark-Evans
R,0.7718
r_obs,23.2031
r_esp,30.0644
z,-8.4660
p,0.0000
area,"1,359,415.7535"


**Interpretación.** Las coordenadas DIVIPOLA (EPSG:4326) son válidas. El panel deja vacíos en la Orinoquía y la Amazonía (sesgo de muestreo espacial) y los municipios están agrupados ($R \approx 0{,}77$).

In [8]:
mostrar(fg.fig_ripley(s["ripley"]))  # Ripley
mostrar(fg.fig_dbscan(mapa))  # DBSCAN
mapa["dbscan"].value_counts().head(8).to_frame("municipios")

,municipios
dbscan,
ruido,193
C3,37
C6,32
C7,23
C11,18
C0,13
C1,10
C5,10


:::{admonition} Detalle: índice de Clark-Evans y función K de Ripley
$R = \bar r_{\text{obs}} / \bar r_{\text{esp}}$ con $\bar r_{\text{esp}} = 0{,}5/\sqrt{n/A}$ (aleatoriedad completa); $K(d) = \frac{A}{n(n-1)}\sum_{i \ne j}\mathbb 1(d_{ij} \le d)$ y $L(d) = \sqrt{K(d)/\pi}$. La envolvente se obtiene con 99 simulaciones de puntos uniformes en la misma envolvente convexa, así que el efecto de borde afecta igual a lo observado y a lo simulado.
:::

In [9]:
display(s["sensibilidad"])  # Moran con distintas matrices de pesos
mostrar(fg.fig_moran(mapa, rezago_espacial(mapa["log_incidencia"])))  # Diagrama de Moran
pd.DataFrame(s["moran"]).T

,sin vecinos,I,z,p
matriz,,,,
k = 4,0,0.3319,9.7818,0.0020
k = 6,0,0.3081,11.0876,0.0020
k = 8,0,0.2754,11.7778,0.0020
k = 12,0,0.2546,14.0761,0.0020
banda 50 km,27,0.3274,9.0605,0.0020
banda 100 km,6,0.2060,10.7724,0.0020
banda 200 km,3,0.1430,13.6026,0.0020


,I,E_I,z,p
log_incidencia,0.3081,-0.0027,11.4422,0.0010
prevalencia,0.0453,-0.0027,1.7035,0.0550


**Interpretación.** La incidencia tiene autocorrelación espacial clara (I ≈ 0,31 con k = 6) y robusta a la matriz de pesos. La proporción de semanas en brote no (I ≈ 0,05, p ≈ 0,06): el canal endémico normaliza el nivel de cada zona.

:::{admonition} Detalle: I de Moran, LISA y Getis-Ord
$I = \frac{n}{S_0}\frac{\mathbf z^\top \mathbf W \mathbf z}{\mathbf z^\top \mathbf z}$ con $E[I] = -1/(n-1)$; significancia por 999 permutaciones. LISA: $I_i = \frac{z_i}{m_2}\sum_j w_{ij} z_j$ con permutación condicional. Getis-Ord: $G_i^* = \frac{\sum_j w_{ij}x_j - \bar x \sum_j w_{ij}}{s\sqrt{(n\sum_j w_{ij}^2 - (\sum_j w_{ij})^2)/(n-1)}}$, que ya es un puntaje z.
:::

In [10]:
mostrar(fg.fig_lisa(mapa))  # LISA
mostrar(fg.fig_gi(mapa))  # Gi*
print(f"LISA significativos tras FDR: {s['fdr_lisa']}")  # Corrección
mapa.loc[mapa["LISA"] == "Alto-Alto", "departamento"].value_counts().to_frame("municipios Alto-Alto")

LISA significativos tras FDR: 53


,municipios Alto-Alto
departamento,
Meta,23
Tolima,22
Huila,12
Putumayo,6
Cundinamarca,5
Santander,4
Casanare,3
Boyacá,1


**Interpretación.** 76 municipios Alto-Alto (Meta, Tolima, Huila, Putumayo) y 53 significativos tras FDR: los focos no son un artefacto de las comparaciones múltiples.

In [11]:
mostrar(fg.fig_vario(s["vario"], float(mapa["log_incidencia"].var())))  # Semivariograma
display(s["heterogeneidad"])  # Por región
pd.Series({"MAUP: correlación temperatura–incidencia (municipios)": s["maup"]["municipios"], "MAUP: (departamentos)": s["maup"]["departamentos"],
           "error mediano de la distancia euclidiana sobre grados": s["error_grados"][0]}).to_frame("valor")

,municipios,prevalencia,AUC lc_m4,AUC oni_l8,AUC temp_anom_m8,AUC prec_total_m8
region,,,,,,
Amazonía,19,0.1421,0.8601,0.6257,0.5164,0.4734
Andina,173,0.0961,0.8760,0.6931,0.5732,0.4685
Caribe,99,0.1023,0.8988,0.6153,0.5279,0.5250
Orinoquía,41,0.1083,0.8588,0.6600,0.5278,0.4869
Pacífica,44,0.1084,0.8912,0.6571,0.6327,0.4634


,valor
MAUP: correlación temperatura–incidencia (municipios),0.1039
MAUP: (departamentos),0.2230
error mediano de la distancia euclidiana sobre grados,0.0020


**Interpretación.** Alcance ≈ 100 km con efecto pepita alto. La relación del clima con el brote cambia entre regiones, y al agregar a departamentos la correlación con la temperatura se duplica (MAUP): se trabaja a escala municipal con haversine.

## Componente espacio-temporal

In [12]:
st = A["espacio_temporal"]  # Espacio-temporal
for a in (2017, 2019):  # Un año sin brotes y uno epidémico
    mostrar(fg.fig_anual_mapa(st["anual"], a))  # Mapa del año
mostrar(fg.fig_focos(st["focos"]))  # Focos
st["focos"]["categoría"].value_counts().to_frame("municipios")

,municipios
categoría,
Nunca foco,193
Esporádico,159
Decreciente,8
Emergente / intensificándose,8
Persistente,8


**Interpretación.** El patrón espacial cambia cada año y los focos se desplazan: 159 municipios son focos esporádicos y solo 8 persistentes.

:::{admonition} Detalle: clasificación de focos
Para cada municipio se calcula $G_i^*$ de la log incidencia de cada año (2015–2021) y la prueba de Mann-Kendall sobre esa serie: **persistente** si es foco ≥ 80 % de los años; **emergente** si es foco el último año con tendencia creciente ($p < 0{,}1$); **decreciente** si fue foco con tendencia decreciente; **esporádico** si fue foco sin tendencia.
:::

In [13]:
mostrar(fg.fig_regiones(st["regiones"]))  # Series por región
display(st["corr_regiones"].round(2))  # Sincronía
st["knox"]  # Prueba de Knox

region,Amazonía,Andina,Caribe,Orinoquía,Pacífica
region,,,,,
Amazonía,1.0000,0.7200,0.5900,0.6000,0.3800
Andina,0.7200,1.0000,0.6400,0.6800,0.4700
Caribe,0.5900,0.6400,1.0000,0.6000,0.0100
Orinoquía,0.6000,0.6800,0.6000,1.0000,0.0300
Pacífica,0.3800,0.4700,0.0100,0.0300,1.0000


,distancia (km),tiempo (semanas),observado,esperado,razon,p,pares_espacio
0,25,2,2278,"2,017.4565",1.1291,0.0010,112907
1,50,4,11127,"8,721.3323",1.2758,0.0010,272238
2,100,4,29314,"24,232.8859",1.2097,0.0010,756176
3,100,8,54888,"45,297.0430",1.2117,0.0010,756176
4,200,8,161579,"144,888.3784",1.1152,0.0010,2419132


**Interpretación.** Las regiones están sincronizadas salvo el Pacífico. La prueba de Knox encuentra en todas las escalas más inicios de brote cercanos en espacio y tiempo de lo esperado (razón 1,1–1,3; p = 0,001).

:::{admonition} Detalle: prueba de Knox
Se cuentan los pares de inicios de brote a menos de $d_s$ km y $d_t$ semanas, y se compara con 999 permutaciones de los tiempos entre lugares (que rompen la interacción y conservan los patrones puramente espaciales y temporales).
:::

In [14]:
mostrar(fg.fig_moran_semanal(st["moran_semanal"]))  # Moran semanal
mostrar(fg.fig_cruzada(st["cruzada"]))  # Vecindario → municipio
st["cruzada"]

,vecinos,propio
k,,
0,0.4343,1.0000
1,0.4242,0.6885
2,0.4189,0.6616
3,0.4123,0.6430
4,0.4055,0.6176
5,0.3952,0.5944
6,0.3863,0.5707
7,0.3753,0.5484
8,0.3637,0.5258


**Interpretación.** La autocorrelación espacial de los casos es positiva casi todas las semanas y el vecindario en $t$ anticipa al municipio en $t+4$ (r ≈ 0,41), aunque menos que su propia historia (r ≈ 0,62). Consecuencia: variables de vecindad rezagadas para los siguientes modelos, y validación por zonas nuevas con zona de separación.

## Preprocesamiento

| Paso | Decisión | Hallazgo que la motiva |
|---|---|---|
| Partición | Cronológica con purga de 4 semanas | Autocorrelación; objetivo en $t+4$ |
| Validación | Ventana creciente por años con purga | Prevalencia y régimen cambian por año |
| Imputación | `temp_missing` + `SimpleImputer(mediana)` | Faltantes MAR estructurales |
| Transformación | $\log(1+x)$ de casos, flujo y población | Asimetría > 14 |
| Escalamiento | `StandardScaler` en el Pipeline | Penalización L2 y escalas distintas |
| Codificación | Seno y coseno de la semana; sin departamento | Estacionalidad cíclica; categorías raras |
| Atípicos | Se conservan | Son epidemias reales |
| Rezagos y ventanas | $t \dots t-3$; medias 4, 8 y 12 | ACF y PACF |
| Anomalías | Respecto a la normal mensual del municipio (2010–2021) | Clima muy distinto entre regiones |
| Regularización | L2 con $C$ por validación cruzada | VIF ≫ 10 |
| Desbalance | `class_weight` en la malla; sin SMOTE | ≈ 10 % de brotes, variable |
| Coordenadas | No se usan como predictoras | Focos que se desplazan |

In [15]:
import inspect  # Código fuente
from e1 import modelo_base as mb  # Modelo base del Primer Entregable
print(inspect.getsource(mb.pipeline_base))  # Pipeline de la regresión logística
print(inspect.getsource(mb.ValidacionAnualPurgada))  # Validación cruzada purgada

def pipeline_base():  # Pipeline del modelo base: imputación + estandarización + regresión logística
    return Pipeline([("imputer", SimpleImputer(strategy="median")),  # Mediana del entrenamiento (protege ante nulos)
                     ("scaler", StandardScaler()),  # Coeficientes comparables y regularización justa
                     ("clf", LogisticRegression(max_iter=5000))])  # Regresión logística con penalización L2

class ValidacionAnualPurgada:  # Validación cruzada hacia adelante por años con purga de etiquetas
    """groups = 10 000 · año de la semana t + año de la semana objetivo (se codifican juntos para GridSearchCV)."""

    def __init__(self, anios=(2018, 2019, 2020, 2021)):  # Años de validación
        self.anios = anios  # Guarda los años

    def split(self, X, y=None, groups=None):  # Índices de cada pliegue
        g = np.asarray(groups)  # Códigos
        anio_t, anio_obj = g // 10000, g % 10000  # Decodifica los dos años
        for a in self.anios:  # Un pli

**Interpretación.** Todo el preprocesamiento ajustable (mediana e imputación, media y desviación) vive dentro del Pipeline, así que en cada pliegue se aprende solo con la parte de entrenamiento. La validación cruzada purgada descarta las filas cuyo objetivo cruza la frontera del año validado.